# ABS Household Spending Indicator 

## Python initialisation

In [1]:
# analytic imports
import pandas as pd
import readabs as ra
from readabs import metacol as mc
import mgplot as mg

# local imports
from abs_helper import get_abs_data

# display charts in this notebook
SHOW = False

## Data Capture

In [2]:
abs_dict, meta, source, _ = get_abs_data("5682.0")

In [3]:
# latest monthly data to ...
print(f"Monthly to: {meta[meta[mc.freq] == "Month"].iloc[0][mc.end].strftime("%B-%Y")}")

# latest quarterly data to (quarter ending) ...
# Quarterly tables only ship with the catalogue when the monthly release lands
# on a quarter-end month; otherwise fall back to the prior quarter-end snapshot.
Q = "Quarter" in meta[mc.freq].values
if Q:
    q_dict, q_meta = abs_dict, meta
else:
    monthly_recent = meta[meta[mc.freq] == "Month"][mc.end].max()
    prev_q_end = pd.Period(monthly_recent, freq="M").asfreq("Q") - 1
    history = prev_q_end.asfreq("M", how="end").strftime("%b-%Y").lower()
    q_dict, q_meta = ra.read_abs_cat(
        "5682.0", history=history, single_excel_only="5682015"
    )
    Q = "Quarter" in q_meta[mc.freq].values

if Q:
    print(f"Quarterly to: {q_meta[q_meta[mc.freq] == "Quarter"].iloc[0][mc.end].strftime("%B-%Y")}")

Monthly to: April-2026


Quarterly to: March-2026


## Monthly headline current prices

In [4]:
def monthly_headline() -> None:
    """Plot the monthly headline series."""

    N = 19
    wanted = [
        "Household spending ;  Total (Household Spending Categories) ;  Australia ;  Current Price ;",
        "Household spending ;  Goods ;  Australia ;  Current Price ;",
        "Household spending ;  Services ;  Australia ;  Current Price ;",
        "Household spending ;  Discretionary ;  Australia ;  Current Price ;",
        "Household spending ;  Non Discretionary ;  Australia ;  Current Price ;",
    ]
    stype = "Seasonally Adjusted"
    cp = "Current Price"
    selector = {
        "Month": mc.freq,
        stype: mc.stype,
    }
    for w in wanted:
        selector[w] = mc.did

        table, series_id, units = ra.find_abs_id(meta, selector)
        series = abs_dict[table][series_id]

        mg.multi_start(
            function=mg.series_growth_plot_finalise,
            data=series,
            starts=(0, -N),
            y0=True,
            title=w.replace(cp, "").replace(" ;", "").strip(),
            lfooter=f"Australia. {cp}. {stype}. ",
            rfooter=source,
            pre_tag="monthly",
            show=SHOW,
        )

        del selector[w]


monthly_headline()

## Quarterly chain volume measures

In [5]:
def quarterly_cvm() -> None:
    """Plot the quarterly CVM series."""
    
    n = 17
    stype = "Seasonally Adjusted"
    price = "Chain Volume Measures"
    table = "5682015"
    selector = {
        table: mc.table,
        stype: mc.stype,
    }
    found = ra.search_abs_meta(q_meta, selector)
    data = q_dict[table]
    for id, row in found.iterrows():
        series = data[id]
        mg.multi_start(
            function=mg.series_growth_plot_finalise,
            data=series,
            starts=(0, -n),
            y0=True,
            title=row[mc.did].replace(" ;", "").replace(price, "").strip(),
            lfooter=f"Australia. {price}. {stype}. ",
            rfooter=source,
            pre_tag="quarterly",
            show=SHOW,
        )

if Q:
    quarterly_cvm()

## Finished

In [6]:
# watermark
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-05-28 21:24:26

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.13.0

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.23
pandas : 3.0.3
readabs: 0.1.9

Watermark: 2.6.0

